In [1]:
!pip install pyspark==3.3.0

/opt/notebooks


In [1]:
import time

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

In [2]:
# Schema for the Kafka JSON message
schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("user_id", IntegerType(), True),
            StructField("username", StringType(), True),
            StructField("password", StringType(), True),
            StructField("email", StringType(), True),
            StructField("created_on", LongType(), True),
            StructField("last_login", LongType(), True)
        ]), True),
        StructField("after", StructType([
            StructField("user_id", IntegerType(), True),
            StructField("username", StringType(), True),
            StructField("password", StringType(), True),
            StructField("email", StringType(), True),
            StructField("created_on", LongType(), True),
            StructField("last_login", LongType(), True)
        ]), True),
        StructField("source", StructType([
            StructField("version", StringType(), True),
            StructField("connector", StringType(), True),
            StructField("name", StringType(), True),
            StructField("ts_ms", LongType(), True),
            StructField("snapshot", StringType(), True),
            StructField("db", StringType(), True),
            StructField("sequence", StringType(), True),
            StructField("ts_us", LongType(), True),
            StructField("ts_ns", LongType(), True),
            StructField("schema", StringType(), True),
            StructField("table", StringType(), True),
            StructField("txId", LongType(), True),
            StructField("lsn", LongType(), True),
            StructField("xmin", LongType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])


In [3]:
spark = SparkSession \
    .builder \
    .appName("Streaming from Kafka") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,com.datastax.spark:spark-cassandra-connector_2.12:3.3.0') \
    .config('spark.cassandra.connection.host', 'cassandra') \
    .config('spark.cassandra.connection.port', '9042') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

In [6]:
streaming_df = spark.readStream\
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "debezium.public.accounts") \
    .option("startingOffsets", "earliest") \
    .load()

json_df = streaming_df.selectExpr("cast(value as string) as value")

# Parse the JSON data
parsedData = json_df.select(from_json(col("value"), schema).alias("data"))

# Flatten the data and select fields
flattenedData = parsedData.select(
    col("data.payload.after.*"),
    col("data.payload.ts_ms"),
    col("data.payload.op")
)


In [ ]:
flattenedData.writeStream\
    .foreachBatch(lambda batch_df, _: batch_df.write \
                  .format("org.apache.spark.sql.cassandra") \
                  .option("keyspace", "db_accounts") \
                  .option("table", "tbl_messages") \
                  .mode("append") \
                  .save()) \
    .start().awaitTermination()